In [12]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
import joblib
import os

In [13]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [19]:
UNLABELLED_PARQUET = "train_unlabelled.parquet"

table = pq.read_table("train_unlabelled.parquet", columns=[])
print(f"Number of rows: {table.num_rows:,}")

unlabelled_count = table.num_rows

Number of rows: 36,029,216


In [21]:
LABELLED_PARQUET = "train_labelled.parquet"

table = pq.read_table(LABELLED_PARQUET, columns=["party", "sentiment"])

labelled_count = table.num_rows

party_counts = {
    str(row["values"].as_py()): row["counts"].as_py()
    for row in pc.value_counts(table["party"])
}

sentiment_by_party = {}
parties = table["party"].unique().to_pylist()
for party in parties:
    mask = pc.equal(table["party"], party) 
    subset = table.filter(mask)

    counts = pc.value_counts(subset["sentiment"])
    sentiment_by_party[party] = {
        str(row["values"].as_py()): row["counts"].as_py()
        for row in counts
    }

print(f"\nLabelled samples: {labelled_count:,}")
print(f"Party distribution: {party_counts}")
for party, dist in sentiment_by_party.items():
    print(f"  {party}: {dist}")



Labelled samples: 864,378
Party distribution: {'republican': 450766, 'democrat': 413612}
  republican: {'positive': 256792, 'negative': 193974}
  democrat: {'positive': 159712, 'negative': 253900}


In [17]:
labelled = pd.read_parquet('train_labelled.parquet', engine='pyarrow')
labelled.head(10)

,clean_text,party,sentiment,timestamp
0,maga maha trumpvance2024 trump2024 magamovement want see hyper inflation going addressed hyper deflationary scarce liquid retaining asset check pin video bio available anyone want asset want learn,republican,positive,2024-10-10
1,favorite post kamala harris pretending like democrat hold crook accountable fucking funny,democrat,positive,2024-10-10
2,ron desantis signed sb304 price gouging back april yeah serious alright like tax tip clown face face tear joy face tear joy face tear joy,democrat,positive,2024-10-10
3,200 people dead hurricane 99 republican voted giving fema money help hurricane victim meanwhile trump billionaire set gofundme account repair mar lago taking away million hurricane victim save mar lago screw america,republican,negative,2024-10-10
4,wtaf coward afraid 60 minute 're incapable telling truth omg would fact checked horror face screaming fear face screaming fear people started fact checking 10 year ago maybe would n't shit show,republican,negative,2024-10-10
5,many u newcomer people turn u asked question good faith shocked u awake pro speech pro america pro peace pro family pro freedom maga movement welcomed u rewarded u maha time flexed biceps medium skin tone nited tate maga maha,republican,positive,2024-10-10
6,member really party thats put american first wake wake one thats still got patriotism democrat party swore follow new communist democrat party change even independed party wake,democrat,positive,2024-10-10
7,shut hell forced woman take experimental vaccine ca n't even define woman clown,democrat,negative,2024-10-10
8,ruined called liar believing hunter biden laptop real liar always lie big time,republican,negative,2024-10-10
9,trust make decision baby body woman issued death certificate child cause death child funeral abused child trust kamala harris index pointing viewer pensive face backhand index pointing left light skin tone,democrat,negative,2024-10-10


In [18]:
PREDICTIONS = "predictions.parquet"

table = pq.read_table(PREDICTIONS, columns=["party", "sentiment"])

count = table.num_rows

party_counts = {
    str(row["values"].as_py()): row["counts"].as_py()
    for row in pc.value_counts(table["party"])
}

sentiment_by_party = {}
parties = table["party"].unique().to_pylist()
for party in parties:
    mask = pc.equal(table["party"], party) 
    subset = table.filter(mask)

    counts = pc.value_counts(subset["sentiment"])
    sentiment_by_party[party] = {
        str(row["values"].as_py()): row["counts"].as_py()
        for row in counts
    }


print(f"Party distribution: {party_counts}")
sentiment_summary = {}
for party, dist in sentiment_by_party.items():
    print(f"  {party}: {dist}")
    mask = df['party'] == party
    if mask.sum() > 0:
        total = mask.sum()
        positive = (df.loc[mask, 'sentiment'] == "positive").sum()
        ratio = positive / total
        sentiment_summary[party] = (positive, total, ratio)
        print(sentiment_summary[party])


Party distribution: {'democrat': 26455060, 'republican': 9574156}
  democrat: {'negative': 14871746, 'positive': 11583314}


NameError: name 'df' is not defined

Democrat:
  Positive sentiment: 11583314/26455060 (43.78%)

Republican:
  Positive sentiment: 5037466/9574156 (52.62%)



In [6]:
df = pd.read_parquet('predictions.parquet', engine='pyarrow')
df.head()

,id,clean_text,text,party,sentiment
0,1844361929781113331,many failure bidens administration directly traced harris deliverable failed win biden mentally deficient president pushed loses biden say told beat trump panicked one debate,"@a_newsman Many failures of Bidens administration are directly traced to Harris on deliverables she failed at. If she wins Biden is the mentally deficient president that had to be pushed out, if she loses Biden says “I told you so. I beat Trump once but you panicked after one debate.”",democrat,negative
1,1844361929550332320,lie biden told american misinformation set exactly 's warning elected suppress free speech,"@atensnut The lies Biden told Americans about ""misinformation"" is a set up to do exactly what she's warning she will do if elected - suppress free speech!",democrat,negative
2,1844361928925384711,someone talented fundraising team,@TheDemocrats here’s someone very talented for your fundraising team.,democrat,positive
3,1844361928585654516,chipper usd card working expressionless face,@LifeOfNapaul Chipper USD card isn’t working 😑,democrat,negative
4,1844361927495123076,yes n't done live interview real journalist since biden dropped n't even stated policy economy,@MTGrepp Yes. She hasn't done a live interview with a real journalist since Biden dropped out. She doesn't even a stated policy on the economy.,democrat,positive


In [6]:
predictions = pd.read_parquet('predictions_with_timestamps.parquet', engine='pyarrow')
predictions.head()

,id,clean_text,text,timestamp,party,sentiment
0,1844361929781113331,many failure bidens administration directly traced harris deliverable failed win biden mentally deficient president pushed loses biden say told beat trump panicked one debate,"@a_newsman Many failures of Bidens administration are directly traced to Harris on deliverables she failed at. If she wins Biden is the mentally deficient president that had to be pushed out, if she loses Biden says “I told you so. I beat Trump once but you panicked after one debate.”",2024-10-10,democrat,negative
1,1844361929550332320,lie biden told american misinformation set exactly 's warning elected suppress free speech,"@atensnut The lies Biden told Americans about ""misinformation"" is a set up to do exactly what she's warning she will do if elected - suppress free speech!",2024-10-10,democrat,negative
2,1844361928925384711,someone talented fundraising team,@TheDemocrats here’s someone very talented for your fundraising team.,2024-10-10,democrat,positive
3,1844361928585654516,chipper usd card working expressionless face,@LifeOfNapaul Chipper USD card isn’t working 😑,2024-10-10,democrat,negative
4,1844361927495123076,yes n't done live interview real journalist since biden dropped n't even stated policy economy,@MTGrepp Yes. She hasn't done a live interview with a real journalist since Biden dropped out. She doesn't even a stated policy on the economy.,2024-10-10,democrat,positive


In [8]:
min_date = predictions.nsmallest(1, 'timestamp')['timestamp'].iloc[0]
max_date = predictions.nlargest(1, 'timestamp')['timestamp'].iloc[0]

print('Minimum date:', min_date)
print('Maximum date:', max_date)


Minimum date: 2009-10-13 00:00:00
Maximum date: 2024-11-30 00:00:00


In [10]:
cutoff_date = pd.Timestamp('2024-05-01')
pre_campaign = predictions[predictions['timestamp'] < cutoff_date].copy()

# Summary
print(f"Number of tweets before May 2024: {len(pre_campaign):,}")

Number of tweets before May 2024: 216,464


In [11]:
pre_campaign.head()


,id,clean_text,text,timestamp,party,sentiment
4321602,1701280839533838398,joe biden spent 11 hiding bunker kamala harris spent boyfriend montel willimas donald trump spent 11 marching war zone fire amp ash army men save american man arena nited tate,Joe Biden spent 9/11 hiding in a bunker.\nKamala Harris spent it with her boyfriend Montel Willimas.\nDonald Trump spent 9/11 marching into a war zone of fire &amp; ashes with an army of his own men to save Americans.\n\nThe Man in the Arena 🇺🇸\n\nhttps://t.co/NzvGOupJ6Y,2023-09-11,democrat,negative
4321603,1701567329237139521,nowhere near time ran presidency 2016 plain much care u still still remember interview oprah asked would ever consider running potus,"@bennyjohnson It was nowhere near the time he ran for the presidency in 2016, but it’s plain to me how much he cares about the US then, and still does. I still remember that interview with Oprah when she asked him if he would ever consider running for POTUS.",2023-09-12,democrat,positive
4321604,1701309669153894814,true leader love nation,@bennyjohnson This is a true leader that loves their nation.,2023-09-11,democrat,positive
4321605,1701348751443382292,said jersey watching thousand muslim celebrating rooftop conveniently forgetting bragged tallest building nyc,@bennyjohnson He said he was in Jersey watching thousands of Muslims celebrating on the rooftops. Are you conveniently forgetting that? Then he bragged about now having the tallest building in NYC.,2023-09-11,democrat,positive
4321606,1701438424757276854,trump true leader american people,@bennyjohnson TRUMP is a TRUE leader of the American people,2023-09-12,democrat,positive
